In [ ]:
import json
import os
from sentence_transformers import SentenceTransformer, CrossEncoder
from qdrant_client import QdrantClient, models
from qdrant_client.models import SparseTextEmbedding
from openai import OpenAI
from deepeval import evaluate
from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    ContextualRecallMetric,
    ContextualPrecisionMetric
)
from deepeval.test_case import LLMTestCase
from deepeval.models.base_model import DeepEvalBaseLLM
from dotenv import load_dotenv

In [ ]:
QDRANT_HOST = "localhost"
QDRANT_PORT = 6333

In [ ]:
client = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)

Model for generting final answer

In [ ]:
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

llm_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

FINAL_ANSWER_MODEL = "google/gemini-3-flash-preview"

In [ ]:
def search(query: str, model: SentenceTransformer, collection: str, initial_k: int = 3):
    print(f"Запит: '{query}'")
    
    query_dense = model.encode(query).tolist()

    search_results = client.query_points(
        collection_name=collection,
        query=query_dense,
        limit=initial_k
    ).points

    if not search_results:
        return "На жаль, за цим запитом нічого не знайдено."

    documents = []
    for hit in search_results:
        text = hit.payload.get("text", "")
        source = hit.payload.get("file_name", "Невідомий документ")
        score = hit.score
        block = f"[Джерело: {source} | Score: {score:.2f}]\n{text}"
        documents.append(block)

    final_context = "\n\n---\n\n".join(documents)
    return final_context

In [ ]:
def generate_final_answer(query: str, retrieved_context: str):
    system_prompt = """
    Ти є інтелектуальним помічником для працівників Українського Католицького Університету (УКУ).
    Твоє завдання — відповісти на запитання користувача, спираючись ВИКЛЮЧНО на наданий контекст з внутрішніх документів.

    ПРАВИЛА:
    1. Якщо в контексті немає відповіді, чесно скажи: "На жаль, у знайдених документах немає інформації для відповіді на це запитання." Не вигадуй жодних фактів.
    2. Формулюй відповідь чітко, професійно та структуровано.
    3. Обов'язково вказуй джерело інформації у форматі: [Джерело: Назва_файлу.pdf].
    """

    user_message = f"КОНТЕКСТ З БАЗИ ДАНИХ:\n{retrieved_context}\n\nЗАПИТАННЯ КОРИСТУВАЧА:\n{query}"

    response = llm_client.chat.completions.create(
        model=FINAL_ANSWER_MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_message}
            ],
            temperature=0.1
    )
    return response.choices[0].message.content

In [ ]:
with open("final_golden_dataset.json", "r", encoding="utf-8") as f:
    golden_data = json.load(f)

In [ ]:
def rag_pipeline(query, model, collection):
    result = search(query, model, collection)
    answer = generate_final_answer(query, result)
    return result, answer

In [ ]:
def create_test_cases(model, collection):
    test_cases = []

    for item in golden_data:
        
        contexts, prediction = rag_pipeline(item["input"], model, collection)
        if isinstance(contexts, str):
            contexts = [contexts]
        contexts = [str(c) for c in contexts]

        test_case = LLMTestCase(
            input=item["input"],
            actual_output=prediction,
            expected_output=item["expected_output"],
            retrieval_context=contexts
        )

        test_cases.append(test_case)

    return test_cases

In [ ]:
class OpenRouterLLM(DeepEvalBaseLLM):

    def __init__(self):
        self.client = OpenAI(
            api_key=OPENROUTER_API_KEY,
            base_url="https://openrouter.ai/api/v1"
        )

    def load_model(self):
        return self.client

    def generate(self, prompt: str):
        response = self.client.chat.completions.create(
            model="openai/gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        return response.choices[0].message.content

    async def a_generate(self, prompt: str):
        return self.generate(prompt)

    def get_model_name(self):
        return "openrouter-gpt"

In [ ]:
judge_model = OpenRouterLLM()

In [ ]:
metrics = [
    AnswerRelevancyMetric(model=judge_model),
    FaithfulnessMetric(model=judge_model),
    ContextualRecallMetric(model=judge_model),
    ContextualPrecisionMetric(model=judge_model)
]

<hr>

# paraphrase-multilingual-MiniLM-L12-v2

In [ ]:
MODEL_BASELINE = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

In [ ]:
COLLECTION_BASELINE = "ucu_documents_baseline"

In [ ]:
evaluate(test_cases=create_test_cases(MODEL_BASELINE, COLLECTION_BASELINE), metrics=metrics)

Evaluate generation while using another method for chunking

In [ ]:
COLLECTION_BASELINE_RECURSIVE_CHAR = "ucu_documents_baseline_recursive_char"

In [ ]:
evaluate(test_cases=create_test_cases(MODEL_BASELINE, COLLECTION_BASELINE_RECURSIVE_CHAR), metrics=metrics)